# GraphDefect-KG — Research Notebook

**Knowledge-Guided Graph Neural Learning for Explainable Industrial Surface Defect Detection**

This notebook is the experimental companion to the GraphDefect-KG backend. It reuses the
*same* feature-extraction, graph-construction and model code that powers the web API
(imported from the `backend` package) so experiments and deployment never diverge.

> **Honesty note.** Every metric, table and figure below is computed at runtime from the
> NEU-DET data on your machine. Nothing is hard-coded. If you run with a small
> `SAMPLES_PER_CLASS` or few `EPOCHS`, treat the numbers as a smoke test, not a
> research result. Scale them up for publishable figures.


## 1–2. Project Overview & Research Objective

Convert a steel surface image into a **region graph** (patch nodes with fused MobileNetV2 +
handcrafted features), learn over it with **GCN / GAT**, fuse with global CNN, handcrafted
and **knowledge-graph** signals, and produce a **graph-grounded explanation**. We compare
classical ML, CNN, graph and hybrid models within one framework.


### Setup — imports and configuration

In [ ]:
import sys, os, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make the backend package importable (run from repo root or notebooks/).
ROOT = Path.cwd()
if (ROOT / "backend").exists():
    PROJECT = ROOT
elif (ROOT.parent / "backend").exists():
    PROJECT = ROOT.parent
else:
    PROJECT = ROOT
sys.path.insert(0, str(PROJECT))
print("Project root:", PROJECT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2

from backend.config import settings, CLASS_NAMES, FOLDER_TO_CLASS
from backend.utils import preprocessing as pp
from backend.utils.feature_extraction import (
    compute_named_features, compute_handcrafted_vector, FEATURE_NAMES)

DEVICE = settings.resolve_device()
print("Device:", DEVICE, "| Classes:", CLASS_NAMES)
print("Dataset images:", settings.dataset_images)

# ---- experiment knobs (increase for research-grade runs) ----
SAMPLES_PER_CLASS = 60     # images per class used in the notebook
EPOCHS = 25
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

## 3. Dataset Loading

In [ ]:
def list_dataset(samples_per_class=SAMPLES_PER_CLASS):
    root = settings.dataset_images
    assert root is not None, "NEU-DET images not found; set NEU_DET_IMAGES."
    items = []
    for folder, label in FOLDER_TO_CLASS.items():
        files = sorted((root/folder).glob("*.jpg"))[:samples_per_class]
        items += [(f, CLASS_NAMES.index(label)) for f in files]
    return items

dataset = list_dataset()
print(f"Loaded {len(dataset)} images across {len(CLASS_NAMES)} classes")

## 4. Dataset Visualization

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, cls in zip(axes, CLASS_NAMES):
    folder = [k for k,v in FOLDER_TO_CLASS.items() if v==cls][0]
    f = sorted((settings.dataset_images/folder).glob("*.jpg"))[0]
    ax.imshow(cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB))
    ax.set_title(cls, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Class Distribution

In [ ]:
import collections
counts = collections.Counter(lbl for _, lbl in dataset)
plt.figure(figsize=(7,3))
plt.bar([CLASS_NAMES[i] for i in range(6)], [counts[i] for i in range(6)], color="#2b6cb0")
plt.xticks(rotation=30, ha="right"); plt.ylabel("images"); plt.title("Class distribution")
plt.tight_layout(); plt.show()

## 6. Image Preprocessing

In [ ]:
sample_img = cv2.imread(str(dataset[0][0]))
pre = pp.preprocess_image(sample_img)
fig, ax = plt.subplots(1, 4, figsize=(14,3.2))
ax[0].imshow(cv2.cvtColor(pre.resized_bgr, cv2.COLOR_BGR2RGB)); ax[0].set_title("Resized")
ax[1].imshow(pre.gray, cmap="gray"); ax[1].set_title("Enhanced gray (CLAHE)")
ax[2].imshow(pre.edges, cmap="gray"); ax[2].set_title("Canny edges")
ax[3].imshow(pre.region_map, cmap="nipy_spectral"); ax[3].set_title("Region map")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("Patches:", len(pre.patches))

## 7. Handcrafted Feature Extraction

In [ ]:
feats = compute_named_features(pre.gray)
for k, v in list(feats.items())[:12]:
    print(f"{k:22s} {v:.4f}")
print("...")
print("Descriptor length:", len(FEATURE_NAMES))

## 8. MobileNetV2 Feature Extraction

In [ ]:
from backend.models.mobilenet_feature_extractor import MobileNetV2FeatureExtractor
extractor = MobileNetV2FeatureExtractor(device=DEVICE)
print(extractor.info())
g_emb = extractor.global_embedding(pre.resized_bgr)
print("Global embedding shape:", g_emb.shape)

## 9. Patch and Region Feature Generation

In [ ]:
from backend.services.graph_service import build_region_nodes
node_x, node_meta, positions, patch_named, patch_cnn = build_region_nodes(pre, extractor)
print("Node feature matrix:", node_x.shape, "| patch CNN:", patch_cnn.shape)
print("Node dim = projected CNN (%d) + handcrafted (%d)" % (settings.model.projected_dim, len(FEATURE_NAMES)))

## 10. KNN Graph Construction

In [ ]:
from backend.models.knn_graph_builder import build_knn_graph
rg = build_knn_graph(node_x, node_meta, positions)
print("nodes:", rg.num_nodes, "edges:", rg.num_edges,
      "| no isolated:", bool((rg.adjacency.sum(1) > 0).all()))

## 11. Graph Visualization

In [ ]:
import networkx as nx
G = nx.Graph()
for i in range(rg.num_nodes):
    G.add_node(i, pos=tuple(positions[i]))
for e in range(rg.edge_index.shape[1]):
    s,t = rg.edge_index[0,e], rg.edge_index[1,e]
    if s!=t: G.add_edge(int(s), int(t), w=float(rg.edge_weight[e]))
pos = {i: (positions[i,0], -positions[i,1]) for i in range(rg.num_nodes)}
plt.figure(figsize=(6,6))
nx.draw(G, pos, node_color="#4299e1", edge_color="#cbd5e0", node_size=280, with_labels=True, font_size=8)
plt.title("KNN region graph (patch adjacency)"); plt.show()

## 12. GCN Implementation (imported from backend)

In [ ]:
from backend.models.gcn_model import GCN
from backend.models.model_loader import NODE_FEATURE_DIM
gcn = GCN(NODE_FEATURE_DIM, num_classes=6).to(DEVICE)
print(gcn)
n_params = sum(p.numel() for p in gcn.parameters())
print("GCN parameters:", n_params)

## 13. GraphSAGE / GAT Implementation

In [ ]:
from backend.models.gnn_model import GNN
gat = GNN(NODE_FEATURE_DIM, arch="gat", num_classes=6).to(DEVICE)
sage = GNN(NODE_FEATURE_DIM, arch="graphsage", num_classes=6).to(DEVICE)
print("GAT params:", sum(p.numel() for p in gat.parameters()))
print("SAGE params:", sum(p.numel() for p in sage.parameters()))

## 14. Knowledge Graph Construction

In [ ]:
from backend.graph.knowledge_graph import KnowledgeGraph
kg = KnowledgeGraph.load_or_build()
for cls in CLASS_NAMES:
    print(f"{cls:16s} -> props {kg.class_properties(cls)} | causes {kg.class_causes(cls)}")

## 15. Knowledge Graph Visualization

In [ ]:
plt.figure(figsize=(11,7))
kgg = kg.g
color = {"defect_class":"#c53030","knowledge_concept":"#00897b","cause_concept":"#00695c"}
node_colors = [color.get(kgg.nodes[n].get("type"), "#888") for n in kgg.nodes]
posk = nx.spring_layout(kgg, seed=SEED, k=0.7)
nx.draw(kgg, posk, node_color=node_colors, with_labels=True, font_size=7,
        node_size=900, edge_color="#cbd5e0")
plt.title("Domain knowledge graph: defects → properties / causes"); plt.show()

## 16. Hybrid Feature Fusion & Graph Dataset

We build a small dataset of region graphs (one per image) plus the auxiliary vectors the
hybrid model consumes. A manual collate function batches variable-size graphs for
graph-level classification without extra dependencies.

In [ ]:
from backend.graph.knowledge_graph import KnowledgeGraph
from dataclasses import dataclass

@dataclass
class Sample:
    x: np.ndarray; edge_index: np.ndarray; edge_weight: np.ndarray
    cnn_global: np.ndarray; cnn_patch: np.ndarray; handcrafted: np.ndarray
    kg_embed: np.ndarray; y: int

def build_sample(path, label):
    img = cv2.imread(str(path))
    pre = pp.preprocess_image(img)
    nx_, meta, posv, pfeats, pcnn = build_region_nodes(pre, extractor)
    rg = build_knn_graph(nx_, meta, posv)
    gfeat = compute_named_features(pre.gray)
    hv = np.array([gfeat[n] for n in FEATURE_NAMES], np.float32)
    return Sample(rg.node_features, rg.edge_index, rg.edge_weight,
                  extractor.global_embedding(pre.resized_bgr),
                  pcnn.mean(0), hv, kg.affinity_vector(gfeat), label)

t0=time.time()
samples = [build_sample(p,l) for p,l in dataset]
print(f"Built {len(samples)} graph samples in {time.time()-t0:.1f}s")

### Train / val / test split and batching

In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(samples)); y_all = np.array([s.y for s in samples])
tr, tmp = train_test_split(idx, test_size=0.3, stratify=y_all, random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.5, stratify=y_all[tmp], random_state=SEED)
print("train/val/test:", len(tr), len(va), len(te))

def collate(indices):
    xs, eis, ews, batch = [], [], [], []
    offset = 0
    for b, i in enumerate(indices):
        s = samples[i]
        xs.append(torch.tensor(s.x))
        eis.append(torch.tensor(s.edge_index) + offset)
        ews.append(torch.tensor(s.edge_weight))
        batch += [b]*s.x.shape[0]; offset += s.x.shape[0]
    return (torch.cat(xs).float().to(DEVICE),
            torch.cat(eis,1).long().to(DEVICE),
            torch.cat(ews).float().to(DEVICE),
            torch.tensor(batch).long().to(DEVICE),
            torch.tensor([samples[i].y for i in indices]).long().to(DEVICE))

## 17. Model Training (GCN / GAT graph-level)

In [ ]:
import torch.nn.functional as F

def train_graph_model(model, epochs=EPOCHS, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(tr)
        for k in range(0, len(perm), 8):
            bx, bei, bew, bb, by = collate(perm[k:k+8])
            opt.zero_grad()
            logits, _ = model(bx, bei, bew, bb)
            loss = F.cross_entropy(logits, by)
            loss.backward(); opt.step()
        if (ep+1) % 5 == 0:
            acc = eval_graph_model(model, va)
            print(f"epoch {ep+1:02d} | val acc {acc:.3f}")
    return model

@torch.no_grad()
def eval_graph_model(model, indices):
    model.eval()
    bx, bei, bew, bb, by = collate(indices)
    pred = model(bx, bei, bew, bb)[0].argmax(1)
    return float((pred == by).float().mean())

gcn = train_graph_model(GCN(NODE_FEATURE_DIM, num_classes=6).to(DEVICE))
gat = train_graph_model(GNN(NODE_FEATURE_DIM, arch="gat", num_classes=6).to(DEVICE))

## 18–19. Validation & Test Evaluation

In [ ]:
from backend.utils.metrics import classification_metrics

@torch.no_grad()
def predict_graph(model, indices):
    model.eval()
    bx, bei, bew, bb, by = collate(indices)
    prob = torch.softmax(model(bx, bei, bew, bb)[0], 1).cpu().numpy()
    return prob.argmax(1), prob, by.cpu().numpy()

for name, m in [("GCN", gcn), ("GAT", gat)]:
    yp, prob, yt = predict_graph(m, te)
    print(name, classification_metrics(yt, yp, prob, labels=list(range(6))))

## 20. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
yp, prob, yt = predict_graph(gat, te)
ConfusionMatrixDisplay.from_predictions(yt, yp, display_labels=CLASS_NAMES,
    xticks_rotation=45, cmap="Blues", normalize=None)
plt.title("GAT — test confusion matrix"); plt.tight_layout(); plt.show()

## 21. Classification Report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(yt, yp, target_names=CLASS_NAMES, zero_division=0))

## 22. ROC and Precision-Recall Curves

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import label_binarize
Yt = label_binarize(yt, classes=list(range(6)))
fig, ax = plt.subplots(1, 2, figsize=(12,4))
for c in range(6):
    if Yt[:,c].sum()==0: continue
    fpr,tpr,_ = roc_curve(Yt[:,c], prob[:,c]); ax[0].plot(fpr,tpr,label=f"{CLASS_NAMES[c]} ({auc(fpr,tpr):.2f})")
    pr,rc,_ = precision_recall_curve(Yt[:,c], prob[:,c]); ax[1].plot(rc,pr,label=CLASS_NAMES[c])
ax[0].plot([0,1],[0,1],"k--",lw=0.7); ax[0].set_title("ROC"); ax[0].legend(fontsize=7)
ax[1].set_title("Precision-Recall"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 23–25. Graph Explainability: node & edge importance

In [ ]:
# GAT attention gives edge importance; aggregate to node importance.
s = samples[te[0]]
x = torch.tensor(s.x).float().to(DEVICE)
ei = torch.tensor(s.edge_index).long().to(DEVICE)
ew = torch.tensor(s.edge_weight).float().to(DEVICE)
gat.eval(); _ = gat(x, ei, ew)
att = gat.edge_importance()
print("edge attention (first 10):", None if att is None else att[:10].detach().cpu().numpy().round(3))

node_imp = np.zeros(s.x.shape[0])
if att is not None:
    a = att.detach().cpu().numpy()
    for e in range(s.edge_index.shape[1]):
        node_imp[s.edge_index[0,e]] += a[e]
node_imp /= (node_imp.max()+1e-9)
plt.figure(figsize=(5,4))
plt.imshow(node_imp.reshape(settings.preprocess.patch_grid, settings.preprocess.patch_grid), cmap="hot")
plt.colorbar(); plt.title("Patch importance heatmap (GAT attention)"); plt.show()

## 26. Knowledge Graph Reasoning Path

In [ ]:
gfeat = compute_named_features(pp.preprocess_image(cv2.imread(str(dataset[te[0]][0]))).gray)
yp0, _, _ = predict_graph(gat, [te[0]])
pred_cls = CLASS_NAMES[int(yp0[0])]
for t in kg.reasoning_path(pred_cls, gfeat):
    print(f"{t['subject']} --{t['relation']}--> {t['object']}  (support={t['support']})")

## 27. Baseline Comparisons (classical ML on MobileNetV2 features)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

Xg = np.stack([s.cnn_global for s in samples]); yg = np.array([s.y for s in samples])
sc = StandardScaler().fit(Xg[tr])
Xtr, Xte = sc.transform(Xg[tr]), sc.transform(Xg[te])
baselines = {
    "KNN": KNeighborsClassifier(5, weights="distance"),
    "SVM": SVC(probability=True),
    "RandomForest": RandomForestClassifier(200, random_state=SEED),
}
try:
    from xgboost import XGBClassifier
    baselines["XGBoost"] = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric="mlogloss")
except Exception: pass

rows = []
for name, clf in baselines.items():
    clf.fit(Xtr, yg[tr]); acc = (clf.predict(Xte)==yg[te]).mean()
    rows.append((name, round(float(acc),3)))
for name, m in [("GCN", gcn), ("GAT", gat)]:
    yp,_,yt = predict_graph(m, te); rows.append((name, round(float((yp==yt).mean()),3)))
import pandas as pd
print(pd.DataFrame(rows, columns=["model","test_accuracy"]).to_string(index=False))

## 28. Ablation Study (component-wise, computed at runtime)

In [ ]:
# Compare: handcrafted-only, CNN-only (KNN), GCN, GAT. Extend as needed.
Xh = np.stack([s.handcrafted for s in samples])
knn_h = KNeighborsClassifier(5).fit(Xh[tr], yg[tr])
abl = [
    ("Handcrafted-only + KNN", (knn_h.predict(Xh[te])==yg[te]).mean()),
    ("MobileNetV2 + KNN", (baselines['KNN'].predict(Xte)==yg[te]).mean()),
    ("GCN (graph)", (predict_graph(gcn, te)[0]==yg[te]).mean()),
    ("GAT (graph)", (predict_graph(gat, te)[0]==yg[te]).mean()),
]
print(pd.DataFrame([(n, round(float(a),3)) for n,a in abl],
                   columns=["configuration","test_accuracy"]).to_string(index=False))

## 29. Error Analysis

In [ ]:
yp, prob, yt = predict_graph(gat, te)
wrong = np.where(yp != yt)[0]
print(f"{len(wrong)} / {len(te)} misclassified")
for w in wrong[:5]:
    print(f"true={CLASS_NAMES[yt[w]]:16s} pred={CLASS_NAMES[yp[w]]:16s} conf={prob[w].max():.2f}")

## 30. Inference on a New Image (full backend pipeline)

In [ ]:
from backend.services.prediction_service import run_prediction
from backend.models.model_loader import load_bundle
_ = load_bundle()  # loads fitted KNN baseline
res = run_prediction(cv2.imread(str(dataset[te[0]][0])), "nb_infer", filename="nb.jpg")
print("Predicted:", res["predicted_class"], "conf", res["confidence"], "source", res["prediction_source"])
print("Explanation:\n", res["explanation"])

## 31. Save Trained Models

In [ ]:
out = settings.saved_models_dir
torch.save({"model": gcn.state_dict()}, out/"gcn_notebook.pt")
torch.save({"model": gat.state_dict()}, out/"gat_notebook.pt")
print("Saved GCN/GAT to", out)
# To make the API use a trained hybrid, train HybridDefectModel similarly and save as
# 'hybrid_model.pt' (state_dict under key 'model'); the loader picks it up automatically.

## 32. Export Frontend-Compatible Graph JSON

In [ ]:
from backend.graph.graph_serializer import build_graph_payload
from backend.graph.graph_reasoner import build_prediction_graph, compute_node_importance_from_patches
img = cv2.imread(str(dataset[te[0]][0])); pre = pp.preprocess_image(img)
nx_, meta, posv, pfeats, pcnn = build_region_nodes(pre, extractor)
rg = build_knn_graph(nx_, meta, posv)
gfeat = compute_named_features(pre.gray)
imp = compute_node_importance_from_patches(pfeats, res["predicted_class"], kg)
pg = build_prediction_graph(region_graph=rg, patch_features=pfeats, global_features=gfeat,
        predicted_class=res["predicted_class"], probabilities=res["probabilities"],
        confidence=res["confidence"], kg=kg, node_importance=imp)
payload = build_graph_payload(pg)
import json as _json
(settings.graph_data_dir/"notebook_graph.json").write_text(_json.dumps(payload, default=float))
print("Exported graph JSON with", payload["counts"], "-> data/graph_data/notebook_graph.json")

## 33. Research Conclusions

Summarise what *your run* showed (fill in from the tables above):

- Relative accuracy of classical ML vs. graph vs. hybrid models on your sample.
- Whether graph models improved over the flat MobileNetV2+KNN baseline.
- Qualitative value of the knowledge-graph reasoning paths and attention heatmaps.

**Do not copy numbers you did not compute.** Increase `SAMPLES_PER_CLASS` and `EPOCHS`,
add cross-validation and multiple seeds before drawing publication-grade conclusions. See
`research/EXPERIMENT_PLAN.md` and `research/ABLATION_STUDY.md` for the full protocol.
